In [12]:
# Import custom mahjong classes
from helper import config
from helper.tile_util import Tile, MahjongMeld
from helper.utility import Hand136, MSPZD, MahjongConverter
from helper.game_util import GameLogEntry, GamePhase
from helper.player import MahjongPlayer
from helper.game import MahjongGame
from helper.visualizer import MahjongReplay, VizMode

from typing import List, Dict, Set, Tuple, Union

from mahjong.hand_calculating.hand import HandCalculator
from mahjong.hand_calculating.hand_config import HandConfig
from mahjong.shanten import Shanten
from mahjong.tile import TilesConverter
from mahjong.agari import Agari

import random
import matplotlib.pyplot as plt
import numpy as np
import copy
import math

In [8]:
def setup_player(name: str) -> Tuple[MahjongPlayer, List[int], List[int], int, List[int]]:
    """
    Setup a single instace of MahjongPlayer object to play a single turn in a single kyoku.
    Returns player, wall, dead_wall, draw_tile, dora_indicators
    """
    player = MahjongPlayer(0, name)
    player.reset_for_kyoku(seat_wind=0, is_dealer=True)
    all_tiles = list(range(136))
    # Allow red dora
    random.shuffle(all_tiles)

    # Reserve dead wall (last 14 tiles)
    dead_wall = all_tiles[-MahjongGame.DEAD_WALL_SIZE:]
    wall = all_tiles[:-MahjongGame.DEAD_WALL_SIZE]

    # give out hand
    player.set_hand([wall.pop() for _ in range(13)])

    draw_tile = wall.pop()

    dora_indicators = [wall.pop()]

    return player, wall, dead_wall, draw_tile, dora_indicators

# Simulating the simulation
- Simulate individual sections / algorithms for solving the rii-chi mahjong game theory
- Below code cell is for checking functionalities
  - Main functionality in mahjong library are...
    - `HandCalculator.estimate_hand_value(**, HandConfig())`
    - `Shanten.calculate_shanten(**, use_chiitoitsu, use_kokushi)`

In [9]:
player, wall, dead_wall, draw_tile, dora_indicators = setup_player("Greedy")
hand_136 = player.hand
draw_136 = Hand136(draw_tile)
hand_34 = hand_136.to_34()
draw_34 = draw_136.to_34()
hand_mspzd = hand_136.to_mspzd(True)
draw_mspzd = draw_136.to_mspzd(True)
# combined len==14
comb_hand_136 = hand_136+draw_136
comb_hand_34 = comb_hand_136.to_34()
comb_hand_mspzd = comb_hand_136.to_mspzd()

print("Current Hand: ", hand_136.ids, "->", hand_mspzd)
print(f"Drawn tile: {draw_136.ids} -> {draw_mspzd}")

hand_calculator = HandCalculator()
config = HandConfig()

result = hand_calculator.estimate_hand_value(
    tiles = comb_hand_136,
    win_tile = draw_tile,
    melds = player._to_library_meld_tuples(),
    dora_indicators = dora_indicators,
    config = config
)
print(f"HandCalculator.estimate_hand_value() = {result}")

# Get Shanten of current hand
shanten = Shanten()
num_shanten = shanten.calculate_shanten( # Calculate minimum shanten[int]
    tiles_34 = comb_hand_34,
    use_chiitoitsu = False,
    use_kokushi = False
)
print(f"Current shanten: Shanten.calculate_shanten() = {num_shanten}")

Current Hand:  [1, 12, 21, 39, 57, 59, 64, 70, 71, 75, 81, 91, 135] -> 146m166899p135sChunz
Drawn tile: [19] -> 5m
HandCalculator.estimate_hand_value() = hand_not_winning
Current shanten: Shanten.calculate_shanten() = 3


In [10]:
# Greedy N-depth search on which hand is best for Shanten
class GreedyShanten():
    def __init__(self, player: MahjongPlayer, wall: List[int]):
        self.shanten = Shanten()
        self.player = player
        self.wall = copy.copy(wall)

    def evaluate_hand(self, hand_136: Hand136) -> Union[int, None]:
        hand_34 = hand_136.to_34()
        if sum(hand_34) == 14:
            return self.shanten.calculate_shanten(
                tiles_34=hand_34,
                use_chiitoitsu=False,
                use_kokushi=False
            )
        
    def search(self, hand_136: Hand136, wall: List[int], depth: int, cur_sim_discard: List = []):
        """
        N-depth greedy search for Shanten efficiency. param wall should be 34 for speed
        """
        if depth == 0:
            return self.evaluate_hand(hand_136)

        best_score = -math.inf

        # try every discard
        for discard in hand_136:
            hand_after_discard = copy.copy(hand_136)
            hand_after_discard.remove(discard)
            cur_sim_discard.append(discard)

            expected_score = 0
            total_tiles = sum(wall)

            # simulate draws
            for tile_type in range(34):

                remaining = wall[tile_type]
                if remaining == 0:
                    continue

                p = remaining / total_tiles

                draw_tile = MahjongConverter.from_34_to_136(tile_type)

                new_hand = hand_after_discard + draw_tile

                new_wall = copy.deepcopy(wall)
                new_wall[tile_type] -= 1

                score = self.search(
                    new_hand,
                    new_wall,
                    depth - 1,
                    cur_sim_discard
                )

                expected_score += p * -score # Negative correlation in EV to Shanten efficiency

            best_score = max(best_score, expected_score)

        return best_score
    
player, wall, dead_wall, draw_tile, dora_indicators = setup_player("Greedy")
greedy_searcher = GreedyShanten(player, wall)
best_score = greedy_searcher.search(copy.copy(player.hand), wall.copy(), 1)